<a href="https://colab.research.google.com/github/alessandronascimento/Prot2Chem_Diffusion/blob/main/notebooks/Prot2ChemDiff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install Prot2ChemDiff
#@markdown Should take ~1 min...
%%time
%%capture
%pip install git+https://github.com/alessandronascimento/Prot2Chem_Diffusion.git

import torch
from tqdm import tqdm
import selfies as sf
from rdkit import Chem
from prot2chemdiff.vae_utils import decode_from_latent
from transformers import AutoTokenizer, AutoModel
import argparse
import sys
from prot2chemdiff.diffuser_lightning import Prot2Chem_Diffusion
from prot2chemdiff.vae_model import MolecularVAE
import pandas as pd
from prot2chemdiff.utils.load_model import load_pretrained_models


def generate_target_embeddings(target_seq='', model_name='facebook/esm2_t33_650M_UR50D', device='cuda'):
    esm_tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    inputs = esm_tokenizer(target_seq, padding=True, truncation=True, max_length=1024, return_tensors='pt').to(device)
    with torch.no_grad():
        prot_outputs = model(**inputs)
        attention_mask = inputs['attention_mask'].unsqueeze(-1)
        prot_embeddings = (prot_outputs.last_hidden_state * attention_mask).sum(1) / attention_mask.sum(1)
    return prot_embeddings

def generate_molecules_batched(
    diffusion_model, vae_model, tokenizer,
    protein_embeddings, target_affinities,
    scale_factor, steps=100, guidance_scale=4.0, device='cuda'
):
    """
    protein_embeddings: Tensor of shape [Batch_Size, 1280]
    target_affinities: Tensor of shape [Batch_Size, 1]
    """
    diffusion_model.eval()
    vae_model.eval()
    diffusion_model.to(device)
    vae_model.to(device)

    batch_size = protein_embeddings.size(0)

    # Setup Conditions for CFG
    protein_cond = protein_embeddings.to(device)
    affinity_cond = target_affinities.to(device)

    uncond_protein = torch.zeros_like(protein_cond).to(device)
    uncond_affinity = torch.zeros_like(affinity_cond).to(device)

    latents = torch.randn((batch_size, 256), device=device)

    scheduler = diffusion_model.scheduler
    scheduler.set_timesteps(num_inference_steps=steps)

    print(f"Generating {batch_size} molecules... (Steps: {steps}, CFG: {guidance_scale})")

    # Denoising Loop
    with torch.no_grad():
        for t in tqdm(scheduler.timesteps, desc="Denoising"):
            t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)

            noise_pred_cond = diffusion_model(latents, t_batch, protein_cond, affinity_cond)

            noise_pred_uncond = diffusion_model(latents, t_batch, uncond_protein, uncond_affinity)

            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)

            latents = scheduler.step(noise_pred, t, latents).prev_sample

    true_latents = latents / scale_factor

    generated_smiles = []
    print("Decoding latents to molecules...")
    with torch.no_grad():
        generated_ids = decode_from_latent(vae_model, true_latents)
        reconstructed_selfies = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        for decoded_selfie in reconstructed_selfies:
            try:
                smiles = sf.decoder(decoded_selfie)
                mol = Chem.MolFromSmiles(smiles)
                if mol is not None:
                    generated_smiles.append(smiles)
                else:
                    generated_smiles.append(f"INVALID_SMILES: {smiles}")
            except Exception:
                generated_smiles.append(f"INVALID_SELFIES: {decoded_selfie}")

    return generated_smiles



In [ ]:
#@title Define parameters and run inference
%%time

seed = 23 # @param {"type":"integer"}
#@markdown *Seed to initialize random generators*

protein_seq = "" # @param {"type":"string"}
#@markdown *Input your protein sequence or leave empty for unconditional generation*

target_affinity = 9.0 # @param {"type":"number"}
#@markdown *Input a -log(Kd) to condition to some affinity.*

batch_size = 128 # @param {"type":"integer"}
#@markdown *Number of molecules to generate.*

guidance_scale = 4.0 # @param {"type":"number"}
#@markdown *Guidance scale for CFG*

steps = 1000 # @param {"type":"integer"}
#@markdown *Number of denoising steps.*

output_prefix = "prot2chemdiff_molecules" # @param {"type":"string"}
#@markdown *Output file prefix*

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(seed)

if protein_seq != "":
  protein_embeddings = generate_target_embeddings(protein_seq)
  desired_affinities = torch.full((batch_size, 1), target_affinity)
else: # Unconditional generation
  print('Running unconditional generation (no protein sequence provided)...')
  protein_embeddings = torch.zeros(1280)
  desired_affinities = torch.zeros(batch_size, 1)

protein_embeddings = protein_embeddings.repeat(batch_size, 1)

SCALE_FACTOR = 2.15709

vae_model, diffusion_model = load_pretrained_models()

tokenizer = AutoTokenizer.from_pretrained("zjunlp/MolGen-large")

new_molecules = generate_molecules_batched(
    diffusion_model, vae_model, tokenizer,
    protein_embeddings, desired_affinities, scale_factor=SCALE_FACTOR,
    guidance_scale=guidance_scale, steps=steps,
    device=device)

df = pd.DataFrame(new_molecules, columns=['generated_smiles'])
df.to_csv(f"{output_prefix}_batch_{batch_size}_guidance_{guidance_scale}.csv")
df

In [ ]:
#@title Show molecules

from rdkit.Chem import PandasTools
PandasTools.InstallPandasTools()
PandasTools.AddMoleculeColumnToFrame(df,'generated_smiles')
df

In [ ]:
#@title Download results

from google.colab import files
files.download(f"{output_prefix}_batch_{batch_size}_guidance_{guidance_scale}.csv")